# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you in loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"\nDataset Title: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Published: {metadata.get('datePublished', 'N/A')}")
print(f"License: {metadata.get('license', 'N/A')}")
print(f"Keywords: {', '.join(metadata.get('keywords', []))}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Show available record sets and their @id
all_record_sets = dataset.record_sets()
print("Available Record Sets (@id):")
for rset in all_record_sets:
    print(f"  {rset['@id']} - {rset.get('name','<Unnamed>')}")

# Let's assume there's one main record set. If there are more, adjust accordingly.
# Print available fields/columns for first record set.
main_record_set_id = all_record_sets[0]['@id'] if all_record_sets else None
if main_record_set_id:
    print(f"\nFields for Record Set {main_record_set_id}:")
    fields = dataset.fields(record_set=main_record_set_id)
    for field in fields:
        print(f"  {field['@id']} - {field.get('name','')} | type: {field.get('dataType','')} | source column: {field.get('column','N/A')}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_sets_ids = [rset['@id'] for rset in dataset.record_sets()]
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns for first record set
if record_sets_ids:
    main_rs_id = record_sets_ids[0]
    print(f"Columns in record set {main_rs_id}:\n{dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field for analysis (e.g., GAD-7 score)
# The actual @id and key usage may vary, so adjust accordingly from the overview.
rs_df = dataframes[main_rs_id]
numeric_field_id = None

# Find a typical numeric field such as GAD-7 or PHQ-9 total score
candidate_fields = ['gad7_score', 'phq9_score', 'psq_score', 'GAD7_total', 'PHQ9_total', 'PSQ_total']
for col in rs_df.columns:
    for cand in candidate_fields:
        if cand.lower() in col.lower():
            numeric_field_id = col
            break
    if numeric_field_id:
        break

if numeric_field_id:
    print(f"Using numeric field: {numeric_field_id}")
else:
    numeric_field_id = rs_df.select_dtypes(include='number').columns[0] if not rs_df.empty else None

threshold = 10  # Example threshold, adjust as appropriate
if numeric_field_id:
    filtered_df = rs_df[rs_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a demographic field, e.g., gender or level_of_education
    group_field_id = None
    for col in rs_df.columns:
        if 'gender' in col.lower():
            group_field_id = col
            break
    if not group_field_id:
        for col in rs_df.columns:
            if 'education' in col.lower():
                group_field_id = col
                break
    
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the numeric field
if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(rs_df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped by a demographic field, show boxplot
    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=rs_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* Using the Croissant schema, we loaded and explored the FAIR² Mental Health Survey Dataset for Kilifi County, Kenya. The dataset covers demographic and mental health indicators, offering scores for GAD-7, PHQ-9, and PSQ.

Key observations:
- The dataset includes primary survey responses and demographic fields.
- Numeric fields such as GAD-7 scores can be used for filtering, normalization, and group-wise analysis.
- Visualization shows a distribution of scores and demographic differences.

Further analysis can focus on cross-relating mental health scores with demographic factors and preparing data for machine learning workflows.

For full transparency, fields, columns, and record sets were always referenced by their `@id` from the Croissant schema.